In [1]:
# Import libraries and set project root path
import pandas as pd
import numpy as np
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Base path:", BASE)

Base path: /Users/alexia/Documents/CASA/Dissertation


In [2]:
# Reload all datasets so this notebook runs independently without 01_data_loading.ipynb

# TS001
ts001 = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/TS001_usual_residents_London_LSOA_2021.csv"), skiprows=7)
ts001 = ts001[ts001["Area"].str.contains("lsoa2021:", na=False)].copy()
ts001["lsoa_code"] = ts001["Area"].str.extract(r"lsoa2021:(E\d+)")
ts001["lsoa_name"] = ts001["Area"].str.extract(r"E\d+ : (.+)$")
ts001 = ts001.drop(columns=["Area"]).rename(columns={"2021": "total_residents"})
ts001 = ts001[["lsoa_code", "lsoa_name", "total_residents"]].reset_index(drop=True)

# TS045
ts045 = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv"), skiprows=7)
ts045 = ts045[ts045["Area"].str.contains("lsoa2021:", na=False)].copy()
ts045["lsoa_code"] = ts045["Area"].str.extract(r"lsoa2021:(E\d+)")
ts045["lsoa_name"] = ts045["Area"].str.extract(r"E\d+ : (.+)$")
ts045 = ts045.drop(columns=["Area"]).rename(columns={
    "No cars or vans in household":        "cars_0",
    "1 car or van in household":           "cars_1",
    "2 cars or vans in household":         "cars_2",
    "3 or more cars or vans in household": "cars_3plus"
})
ts045 = ts045[["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]].reset_index(drop=True)

# IMD 2019
xl = pd.ExcelFile(os.path.join(BASE, "03_data/demand/imd2019/IMD2019_LSOA_England.xlsx"))
imd = xl.parse("IMD2019").rename(columns={
    "LSOA code (2011)":                           "lsoa_code_2011",
    "LSOA name (2011)":                           "lsoa_name",
    "Local Authority District code (2019)":       "lad_code",
    "Local Authority District name (2019)":       "lad_name",
    "Index of Multiple Deprivation (IMD) Rank":   "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile": "imd_decile"
})

# OpenStreetEV GLA
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))
osev = osev_raw[["country_co","id","external_l","name","address","postal_cod","state","coordinate","coordina_1","location_c","location_1"]].copy()
osev = osev.rename(columns={
    "country_co": "country_code", "id": "location_id", "external_l": "external_uuid",
    "name": "location_name", "address": "address", "postal_cod": "postcode",
    "state": "borough", "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "location_1": "location_type"
})

# GLA Join, EVSE Status, Device UK
gla_join    = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/gla_location_evse_join.csv"))
evse_status = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/evse_status_gla.csv"))
device      = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/device_all_uk.csv"), low_memory=False)

print("All datasets reloaded successfully.")

All datasets reloaded successfully.


In [3]:
# Create output directory for cleaned data if it does not exist
output_dir = os.path.join(BASE, "05_processed")
os.makedirs(output_dir, exist_ok=True)
print("Output directory:", output_dir)

Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed
